# Notebook 03: Feature Representation & Semantic Space

**ComplaintScope: Consumer-Complaint Triage and Early Warning System**  
**Course:** CBS3009 – Advanced Social, Text and Media Analytics  
**Authors:** Ruhaan bansal (23BBS0046), Vedant Rajesh (23BBS0037), Anish Desai (23BBS0119)  
**Faculty:** Sathish Kumar K | **Institution:** VIT Vellore

### Objectives:
1. Extract lexical features via CountVectorizer (BoW) and sublinear TfidfVectorizer fit strictly on the chronological training split.
2. Empirically validate Zipf's Law on token frequency distributions.
3. Identify top discriminative terms per product class using mean TF-IDF weights.
4. Generate dense semantic embeddings via sentence-transformers (`all-MiniLM-L6-v2`).
5. Quantitatively evaluate semantic retrieval accuracy using Cosine kNN Precision@k (k = 1, 3, 5, 10).
6. Construct 2D dimensionality reduction projections (UMAP / PCA) to inspect class separability.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()))

import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.config import load_config, get_path
from src.features import evaluate_knn_precision_at_k, build_metadata_features

print('Setup complete. All feature modules imported successfully.')

## 1. Lexical Representations (BoW & TF-IDF)

To prevent lookahead and vocabulary contamination, vectorizers are fit exclusively on the chronological training set ($N = 13,983$). The vocabulary size is capped at 30,000 features with sublinear term-frequency scaling ($1 + \log(\text{tf})$) and bigrams ($1, 2$). Validation and test sets are strictly transformed using the frozen training vocabulary.

In [ ]:
bow_path = Path('../data/processed/bow_vectorizer.joblib') if Path('../data/processed/bow_vectorizer.joblib').exists() else Path('data/processed/bow_vectorizer.joblib')
tfidf_path = Path('../data/processed/tfidf_vectorizer.joblib') if Path('../data/processed/tfidf_vectorizer.joblib').exists() else Path('data/processed/tfidf_vectorizer.joblib')

bow_vec = joblib.load(bow_path)
tfidf_vec = joblib.load(tfidf_path)

print(f'BoW Vocabulary Size:      {len(bow_vec.vocabulary_):,}')
print(f'TF-IDF Vocabulary Size:   {len(tfidf_vec.vocabulary_):,}')
print(f'Sublinear TF scaling:     {tfidf_vec.sublinear_tf}')
print(f'N-gram range:             {tfidf_vec.ngram_range}')

## 2. Empirical Verification of Zipf's Law

Zipf's law states that the frequency of any word is inversely proportional to its rank in the frequency table ($f(r) \propto 1/r^s$, where $s \approx 1$). In log-log space, this manifests as a linear decay.

In [ ]:
zipf_img = Path('../results/figures/fig07_zipf_distribution.png') if Path('../results/figures/fig07_zipf_distribution.png').exists() else Path('results/figures/fig07_zipf_distribution.png')
if zipf_img.exists():
    from IPython.display import Image, display
    display(Image(filename=str(zipf_img), width=700))
else:
    print('Zipf figure not found at', zipf_img)

## 3. Class-Specific Top Discriminative Terms

By computing the mean TF-IDF weight vector for each product category across the training partition, we extract the top-15 most characteristic terms for each class.

In [ ]:
terms_img = Path('../results/figures/fig08_class_top_terms.png') if Path('../results/figures/fig08_class_top_terms.png').exists() else Path('results/figures/fig08_class_top_terms.png')
if terms_img.exists():
    from IPython.display import Image, display
    display(Image(filename=str(terms_img), width=950))
else:
    print('Class terms figure not found at', terms_img)

## 4. Dense Semantic Embeddings & kNN Retrieval

We map each complaint narrative to a 384-dimensional dense semantic embedding using `all-MiniLM-L6-v2`. Using cosine similarity in the normalized embedding space, we evaluate quantitative retrieval accuracy (Precision@k) on the validation set against training corpus targets.

In [ ]:
knn_path = Path('../results/tables/knn_precision_at_k.json') if Path('../results/tables/knn_precision_at_k.json').exists() else Path('results/tables/knn_precision_at_k.json')
if knn_path.exists():
    with open(knn_path) as f:
        knn_metrics = json.load(f)
    print('Cosine kNN Retrieval Performance (Precision@k):')
    for k_metric, val in knn_metrics.items():
        print(f'  {k_metric}: {val*100:.2f}%')
else:
    print('kNN metrics file not found.')

## 5. 2D Manifold Projection

Visualizing the high-dimensional MiniLM embeddings via 2D manifold learning (UMAP/PCA) demonstrates cluster coherence and category separability across consumer grievance domains.

In [ ]:
proj_img = Path('../results/figures/fig09_2d_projection.png') if Path('../results/figures/fig09_2d_projection.png').exists() else Path('results/figures/fig09_2d_projection.png')
if proj_img.exists():
    from IPython.display import Image, display
    display(Image(filename=str(proj_img), width=850))
else:
    print('2D projection figure not found at', proj_img)

## 6. Feature Representation Summary & Quality Gate Verification

- **BoW & TF-IDF:** Preserved strictly without val/test leakage; 30,000 features saved to `data/processed/`.
- **MiniLM Embeddings:** 384-dimensional dense normalized representations cached to `data/processed/emb_*.npy`.
- **kNN Evaluation:** Retrieval precision quantitatively validated across multiple k thresholds.
- **Downstream Readiness:** Features cached for immediate ingestion by Component 1 (Product Classification) and Component 2 (Relief Risk Prediction).